# NLP Basics: From Messy Text to Clean Tokens

This notebook covers the fundamental steps in Natural Language Processing (NLP) corresponding to Levels 1-7 of the NLP Game.

**Goals:**
1.  **Noise Removal:** Cleaning up messy text.
2.  **Tokenization:** Splitting text into words (Simple vs. BERT).
3.  **Stop Word Removal:** Focusing on important words.
4.  **Stemming vs. Lemmatization:** Reducing words to their root form.
5.  **Vectorization (Bag of Words):** Turning text into numbers.
6.  **Vector Embeddings:** Capturing meaning with numbers.
7.  **Padding:** Making sequences uniform for the model.

## Level 1: Noise Removal

Real-world text is messy! Before we can analyze it, we need to normalize it.

**Steps:**
1.  Convert to **Lowercase** (so "Apple" and "apple" are the same).
2.  Remove **Punctuation** (like "!", "...", "?").

In [15]:
import string

text = "Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed"

# 1. Lowercase
text_lower = text.lower()

# 2. Remove Punctuation using string translation
# This creates a table that swaps punctuation for 'Nothing'
#First argument → characters to replace
#Second argument → replacement
#Third argument → characters to remove

translator = str.maketrans('', '', string.punctuation)
clean_text = text_lower.translate(translator)

print(string.punctuation)
print(" ")
print("Original:", text)
print("Cleaned: ", clean_text)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
 
Original: Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed
Cleaned:  ugh the deliveries were delayed i seriously hate waiting annoyed


In [14]:
table = str.maketrans("aeiou", "*****")
text = "hello world"

print(text.translate(table))

h*ll* w*rld


## Level 2: Tokenization

Computers can't read sentences. We need to chop them into individual pieces called **Tokens**.

### Method A: Simple Split (The Chopper)
Just splits by spaces. Simple but misses nuance.

In [3]:
# The simple way (split by space)
tokens = clean_text.split()

print("Tokens:", tokens)
print("Token Count:", len(tokens))

Tokens: ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']
Token Count: 10


### Method B: BERT Tokenizer (The Smart Reader)
Used by modern AI (like BERT/Transformers). It breaks complex words into sub-word pieces (like Lego bricks) to understand them better.

In [ ]:
!pip install transformers

In [4]:
# !pip install transformers
from transformers import AutoTokenizer

# We will use the BERT tokenizer (Standard in industry)
# This requires internet to download the vocab
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
complex_text = "The microtransactional system was counterintuitive."

# Output will likely split 'microtransactional' into ['micro', 'transaction', '##al']
print("BERT Tokens:", tokenizer.tokenize(complex_text))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BERT Tokens: ['the', 'micro', '##tra', '##ns', '##act', '##ional', 'system', 'was', 'counter', '##int', '##uit', '##ive', '.']


## Level 3: Stop Word Removal

Some words appear everywhere but carry little meaning (e.g., "the", "is", "at"). We remove them to focus on the keywords.

In [6]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:
import nltk
from nltk.corpus import stopwords

# nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

filtered_tokens = [word for word in tokens if word not in stop_words]

print("Input:", tokens)
print("Result:", filtered_tokens)

Input: ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']
Result: ['ugh', 'deliveries', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed']


## Level 4: Stemming vs. Lemmatization

Now we need to handle word variations (like "running" vs "run").

*   **Stemming:** Chops off the end (fast but crude).
*   **Lemmatization:** Uses a dictionary to find the root word (slower but accurate).

In [8]:
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet')
# nltk.download('omw-1.4') # Often needed for newer NLTK versions

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["deliveries", "waiting", "delayed", "studies"]

for w in words:
    # Notice: Stemmer makes "studi", Lemmatizer finds "study"
    print(w, "|", stemmer.stem(w), "|", lemmatizer.lemmatize(w))

[nltk_data] Downloading package wordnet to /root/nltk_data...


deliveries | deliveri | delivery
waiting | wait | waiting
delayed | delay | delayed
studies | studi | study


## Level 5: Vectorization (Bag of Words)

Text is still text. Models need numbers! We use **Bag of Words** to count how often each word appears.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# The "Corpus" (Our collection of sentences)
corpus = [
    "Win free money",
    "Win huge money",
    "The deliveries were delayed"
]

# Create the Bag of Words Model
vectorizer = CountVectorizer()

# Learn vocabulary and transform the corpus into vectors
X = vectorizer.fit_transform(corpus)

# Get the feature names (vocabulary)
vocab = vectorizer.get_feature_names_out()

# Show the Vectors (as a DataFrame for better visibility)
df = pd.DataFrame(X.toarray(), columns=vocab, index=corpus)

print("Vocabulary:", vocab)
print("\nBag of Words Matrix:")
print(df)

Vocabulary: ['delayed' 'deliveries' 'free' 'huge' 'money' 'the' 'were' 'win']

Bag of Words Matrix:
                             delayed  deliveries  free  huge  money  the  \
Win free money                     0           0     1     0      1    0   
Win huge money                     0           0     0     1      1    0   
The deliveries were delayed        1           1     0     0      0    1   

                             were  win  
Win free money                  0    1  
Win huge money                  0    1  
The deliveries were delayed     1    0  


## Level 6: Vector Embeddings (Sentence Transformers)

Bag of Words doesn't understand meaning (it thinks "happy" and "joy" are different). Use **Embeddings** to capture meaning in numbers.

In [10]:
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer

# 1. Download the Brain (Pre-trained on billions of sentences)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Give it words
words = ["King", "Queen", "Apple"]
embeddings = model.encode(words)

# 3. See the Vectors (The DNA)
for i in range(len(words)):
    print("Word:", words[i])
    print("Vector:", embeddings[i][:3]) # Show just first 3 numbers

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Word: King
Vector: [-0.05959929  0.0505124  -0.06951005]
Word: Queen
Vector: [ 0.03548702 -0.06560463 -0.00993493]
Word: Apple
Vector: [-0.00613845  0.03101176  0.06479359]


## Level 7: Padding

Models like neat rectangles (matrices). We use **Keras** to automate the tokenization and padding.

In [11]:
# pip install tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

sentences = [
    "I love AI",
    "The deliveries were delayed",
    "Win free money"
]

# 1. Tokenize (Assign numbers)
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)

print("Sequences:", sequences)

# 2. Pad (Add zeros to make them same length)
# padding='post' adds zeros at the END
padded = pad_sequences(sequences, padding='post')

print("\nPadded Matrix:")
print(padded)

Sequences: [[2, 3, 4], [5, 6, 7, 8], [9, 10, 11]]

Padded Matrix:
[[ 2  3  4  0]
 [ 5  6  7  8]
 [ 9 10 11  0]]


### Conclusion
Now every sentence is the same length (a perfect Matrix). We are ready to train! 🚀